# Deep Past Initiative – Machine Translation (Training Notebook)

This notebook is a **pretrain notebook** for this Kaggle competition.

Main ideas:
- Use **ByT5** to handle noisy Akkadian transliterations at the character level
- Pre-train the model using Span Corruption on un-translated Akkadian text
- Later you can use the model you produce to train on Akkadian-English translation pairs.

Bibliography here: https://chatgpt.com/share/69b302c0-46e4-8007-9bd4-94ed3efe88c0

In [1]:
!pip install evaluate sacrebleu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 10.5 MB/s eta 0:00:00


In [2]:
import os
import gc
import re
import pandas as pd
import numpy as np
import torch
from sklearn.model_selection import train_test_split
from sklearn.metrics.pairwise import cosine_similarity
from datasets import Dataset
from transformers import (
    AutoTokenizer, 
    AutoModelForSeq2SeqLM, 
    DataCollatorForSeq2Seq, 
    Seq2SeqTrainingArguments, 
    Seq2SeqTrainer
)
from sentence_transformers import SentenceTransformer, util
import evaluate
from tqdm import tqdm

2026-03-17 23:27:48.633294: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1773790068.809766      23 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1773790068.867529      23 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1773790069.316586      23 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1773790069.316628      23 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1773790069.316631      23 computation_placer.cc:177] computation placer alr

In [3]:
class Config:
    # Akkadian transliteration contains a lot of noise and many unknown words, so
    # ByT5, which processes text at the character (byte) level rather than the word level, is the strongest choice.
    MODEL_NAME = "google/byt5-small" 
    # MODEL_NAME = "/kaggle/input/notebooks/shwesh/dpc-starter-train/byt5-akkadian-model/" 
    # MODEL_NAME = "/kaggle/input/notebooks/shwesh/dpc-mask-fill-pretrain-3-englishless/byt5-akkadian-model/"
    
    # ByT5 tends to produce longer token sequences, but 512 tokens is enough at the sentence level.
    MAX_LENGTH = 512
    
    BATCH_SIZE = 8       # Adjust depending on GPU memory (on a P100 you can usually go with 8–16).
    EPOCHS = 20
    LEARNING_RATE = 2e-4
    OUTPUT_DIR = "./byt5-akkadian-model"
    SPAN_MEAN = 20

In [4]:
# Fix the seed (for reproducibility).
def seed_everything(seed=42):
    torch.manual_seed(seed)
    np.random.seed(seed)
    torch.cuda.manual_seed(seed)
    
seed_everything()

In [5]:
INPUT_DIR = "/kaggle/input/competitions/deep-past-initiative-machine-translation"

In [6]:
train_df = pd.read_csv("/kaggle/input/notebooks/shwesh/data-exploration/extensive_monolingual_train.csv")
test_df =  pd.read_csv("/kaggle/input/notebooks/shwesh/data-exploration/extensive_monolingual_test.csv")


In [7]:
print(f"Original Train Data: {len(train_df)} docs")

Original Train Data: 7515 docs


In [8]:
def simple_sentence_aligner(df):
    """
    【戦略の肝】
    Trainデータの「文書(複数文)」を、Testデータと同じ「文(1文)」に分割します。
    ここでは「英語の文数」と「アッカド語の行数」が一致する場合のみ分割する
    というヒューリスティック（簡易ルール）を使います。
    """
    aligned_data = []
    
    for idx, row in df.iterrows():
        src = str(row['transliteration'])
        tgt = str(row['translation'])
        
        # Split the English text by sentence-ending punctuation.
        tgt_sents = [t.strip() for t in re.split(r'(?<=[.!?])\s+', tgt) if t.strip()]
        
        # Assume the Akkadian text is often separated by newlines and split accordingly.
        src_lines = [s.strip() for s in src.split('\n') if s.strip()]
        
        # If the counts match, trust it as 1-to-1 pairs and use the split version.
        if len(tgt_sents) > 1 and len(tgt_sents) == len(src_lines):
            for s, t in zip(src_lines, tgt_sents):
                if len(s) > 3 and len(t) > 3: # Remove junk/noisy data.
                    aligned_data.append({'transliteration': s, 'translation': t})
        else:
            # If splitting fails (counts don't match), keep the original document pair as-is (safe fallback).
            aligned_data.append({'transliteration': src, 'translation': tgt})
            
    return pd.DataFrame(aligned_data)


In [9]:
def select_corruption_mask(
    seq_length, corruption_rate=0.15, mean_span_length=3
):
    """
    Create a boolean mask indicating which positions to corrupt.
    Spans are selected to achieve approximately the target corruption rate.
    """
    # Calculate expected number of spans
    num_tokens_to_corrupt = int(seq_length * corruption_rate)
    num_spans = max(1, num_tokens_to_corrupt // mean_span_length)

    # Sample span lengths from geometric distribution
    p = 1 / mean_span_length
    span_lengths = np.random.geometric(p, size=num_spans)

    # Clip to ensure we don't exceed the corruption budget
    total_corrupted = span_lengths.sum()
    if total_corrupted > num_tokens_to_corrupt:
        # Scale down span lengths proportionally
        scale = num_tokens_to_corrupt / total_corrupted
        span_lengths = np.maximum(1, (span_lengths * scale).astype(int))

    # Randomly place spans without overlap
    mask = np.zeros(seq_length, dtype=bool)
    available_positions = list(range(seq_length))

    for length in span_lengths:
        if len(available_positions) < length:
            break
        # Choose a starting position
        valid_starts = [i for i in range(len(available_positions) - length + 1)]
        if not valid_starts:
            break
        start_idx = np.random.choice(valid_starts)

        # Mark positions as corrupted
        for offset in range(length):
            pos = available_positions[start_idx]
            mask[pos] = True
            available_positions.remove(pos)

    return mask
def get_span_boundaries(mask):
    """
    Extract start and end indices for each contiguous span in the mask.
    Returns list of (start, end) tuples where end is exclusive.
    """
    spans = []
    in_span = False
    start = 0

    for i, is_corrupted in enumerate(mask):
        if is_corrupted and not in_span:
            # Starting a new span
            start = i
            in_span = True
        elif not is_corrupted and in_span:
            # Ending current span
            spans.append((start, i))
            in_span = False

    # Handle span at end of sequence
    if in_span:
        spans.append((start, len(mask)))

    return spans
def corrupt_sequence(tokens, mask, sentinel_prefix="<extra_id_"):
    """
    Apply span corruption to a token sequence.

    Args:
        tokens: List of tokens
        mask: Boolean array indicating corrupted positions
        sentinel_prefix: Prefix for sentinel tokens

    Returns:
        corrupted_input: Input sequence with sentinels replacing spans
        target: Target sequence for reconstruction
    """
    spans = get_span_boundaries(mask)

    # Build corrupted input
    corrupted_input = []
    last_end = 0

    for span_idx, (start, end) in enumerate(spans):
        # Add uncorrupted tokens before this span
        corrupted_input.extend(tokens[last_end:start])
        # Add sentinel for this span
        corrupted_input.append(f"{sentinel_prefix}{span_idx}>")
        last_end = end

    # Add remaining uncorrupted tokens
    corrupted_input.extend(tokens[last_end:])

    # Build target sequence
    target = []
    for span_idx, (start, end) in enumerate(spans):
        # Add sentinel
        target.append(f"{sentinel_prefix}{span_idx}>")
        # Add original span tokens
        target.extend(tokens[start:end])

    return corrupted_input, target


def span_corruption(df):
    #make a new df with "source" and "target" columns.

    sources = []
    targets = []
    
    for idx, row in tqdm(df.iterrows()):
        akk = str(row['transliteration'])
        # idk how tokenizer works. I'm just gonna throw raw text in and assume it works out????? TODO: Read byt5 paper to see if this is bad:
        mask = select_corruption_mask(len(akk),mean_span_length=Config.SPAN_MEAN)
        src, tgt = corrupt_sequence(akk, mask)
        src = "".join(src)
        tgt = "".join(tgt)
        sources.append(src)
        targets.append(tgt)

    # Return new df
    return pd.DataFrame({'source':sources,'target':targets})

In [10]:
test_df = span_corruption(test_df)
test_split_dataset = Dataset.from_pandas(test_df)
print(f"Masked Test Data: {len(test_df)} examples (Mask applied)")
print(test_df.head())

834it [00:00, 1840.66it/s]

Masked Test Data: 834 examples (Mask applied)
                                              source  \
0  a-na a-šùr-na-da ù a-šùr-ta-ak-lá-ku qí-bi4-ma...   
1  a-na pu-šu-ke-en6 qí-bi-ma um-ma ta-ri-iš-ma-t...   
2  15 ma-na KÙ.BABBAR ṣa-ru-pá-am i-ṣé-er DINGIR-...   
3   um-ma da-ki-ip-ša-ri-ma a-na ú-ru-mu a-šar uš...   
4  -tim nu-qá-dí-ìš ṭup-pá-áš-nu ba-ab DINGIR ni-...   

                                              target  
0  <extra_id_0>TÚG}ku-ta-nu 18 TÚG šu-ru-tum 2 {T...  
1  <extra_id_0>ri-i<extra_id_1>a-<extra_id_2>ší l...  
2  <extra_id_0>-al-ú-tim la iš-qúl 1.5 GÍN.TA a-n...  
3                     <extra_id_0>-ak-ni-ša-ri a-ḫi-  
4  <extra_id_0>ma-da-a<extra_id_1>-li a-ta a-lá-n...  


In [11]:
# For loop
gc.collect()
torch.cuda.empty_cache()
model = AutoModelForSeq2SeqLM.from_pretrained(Config.MODEL_NAME)#, local_files_only=True)
tokenizer = AutoTokenizer.from_pretrained(Config.MODEL_NAME)#, local_files_only=True)
data_collator = DataCollatorForSeq2Seq(tokenizer, model=model)



for i in range(Config.EPOCHS):
    print(f"epoch:{i}")
    # Convert to fill-mask

    train_expanded = span_corruption(train_df)

    train_split_dataset = Dataset.from_pandas(train_expanded)

    # After splitting, the keys are 'train' and 'test' (we'll use 'test' as validation).

    # ==========================================
    # 3. Tokenization & preprocessing
    # ==========================================
    

    # Fix the corresponding section in dpc-starter-train.
    PREFIX = "Fill masked Akkadian: "

    def preprocess_function(examples):
        inputs = [PREFIX + str(ex) for ex in examples["source"]]
        targets = [str(ex) for ex in examples["target"]]
        
        model_inputs = tokenizer(inputs, max_length=Config.MAX_LENGTH, truncation=True)
        labels = tokenizer(targets, max_length=Config.MAX_LENGTH, truncation=True)
        
        model_inputs["labels"] = labels["input_ids"]
        return model_inputs

    tokenized_train = train_split_dataset.map(preprocess_function, batched=True)
    tokenized_val = test_split_dataset.map(preprocess_function, batched=True)


    # ==========================================
    # 4. Model training (fine-tuning)
    # ==========================================
    gc.collect()
    torch.cuda.empty_cache()

    # Metric (chrF++ is part of the competition metric and measures character-level precision/overlap).
    metric = evaluate.load("chrf")

    def compute_metrics(eval_preds):
        preds, labels = eval_preds
        if isinstance(preds, tuple): preds = preds[0]
        try:
            decoded_preds = tokenizer.batch_decode(preds, skip_special_tokens=True)
        except:
            print(f"The bad preds are: {preds}")
            print(f"with type:{preds.__class__.__name__}")
            print("Ignoring computing metrics and continuing onward")
            return {"chrf": 0} 
        # Ignore -100 in the labels.
        labels = np.where(labels != -100, labels, tokenizer.pad_token_id)
        decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)
        
        decoded_preds = [pred.strip() for pred in decoded_preds]
        decoded_labels = [[label.strip()] for label in decoded_labels]
        
        result = metric.compute(predictions=decoded_preds, references=decoded_labels)
        return {"chrf": result["score"]}

    args = Seq2SeqTrainingArguments(
        output_dir=Config.OUTPUT_DIR,
        eval_strategy="epoch",
        save_strategy="epoch",
        learning_rate=Config.LEARNING_RATE,
        
        # === Key fixes ===
        fp16=False,                     # ★Set to False to prevent a NaN error (required).
        per_device_train_batch_size=4,  # ★fp32 uses more memory, so reduce the batch size (8 -> 4).
        per_device_eval_batch_size=4,
        gradient_accumulation_steps=2,  # ★To compensate, accumulate gradients to keep the effective batch size at 8.
        # ======================
        
        weight_decay=0.01,
        save_total_limit=2,
        num_train_epochs=1,
        predict_with_generate=True,
        logging_steps=10,               # Inspect logs in more detail.
        report_to="none"
    )

    trainer = Seq2SeqTrainer(
        model=model,
        args=args,
        train_dataset=tokenized_train,
        eval_dataset=tokenized_val,
        data_collator=data_collator,
        tokenizer=tokenizer,
        compute_metrics=compute_metrics
    )

    print("Starting Training (FP32 mode)...")
    trainer.train()



config.json:   0%|          | 0.00/698 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/1.20G [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.20G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

...

Map:   0%|          | 0/7515 [00:00<?, ? examples/s]

Map:   0%|          | 0/834 [00:00<?, ? examples/s]

/tmp/ipykernel_23/4188218771.py:92: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Seq2SeqTrainer.__init__`. Use `processing_class` instead.
  trainer = Seq2SeqTrainer(


Starting Training (FP32 mode)...


Epoch,Training Loss,Validation Loss,Chrf
1,1.857700,1.624819,9.158284


In [12]:
# --- Save Model ---
# Important: the model saved here will be loaded in the next notebook.
trainer.save_model(Config.OUTPUT_DIR)
tokenizer.save_pretrained(Config.OUTPUT_DIR)
print(f"Model saved to {Config.OUTPUT_DIR}")


Model saved to ./byt5-akkadian-model
